# 4M: Massively Multimodal Masked Modeling 教程

本教程介绍 4M 风格的多模态统一模型。

---

## 环境设置

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. 4M 模型架构概述

In [ ]:
from fourm import FourMConfig, create_fourm_model

config = FourMConfig()
print('4M 默认配置:')
print(f'  image_size: {config.image_size}')
print(f'  codebook_size: {config.codebook_size}')
print(f'  d_model: {config.d_model}')
print(f'  modalities: {config.modalities}')

In [ ]:
# 创建不同大小的模型
print('不同大小的 4M 模型:')
for size in ['tiny', 'small', 'base']:
    model = create_fourm_model(size)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  {size}: d_model={model.config.d_model}, params={n_params/1e6:.1f}M')

## 2. 向量量化 (Vector Quantization)

In [ ]:
from fourm import VectorQuantizer

vq = VectorQuantizer(codebook_size=512, codebook_dim=64)
z = torch.randn(2, 8, 8, 64)
quantized, indices, vq_loss = vq(z)

print(f'输入形状: {z.shape}')
print(f'量化后形状: {quantized.shape}')
print(f'索引形状: {indices.shape}')
print(f'VQ 损失: {vq_loss.item():.4f}')

In [ ]:
# 可视化 codebook 使用
indices_flat = indices.flatten().numpy()
plt.figure(figsize=(10, 4))
plt.hist(indices_flat, bins=50)
plt.xlabel('Codebook Index')
plt.ylabel('Frequency')
plt.title('Codebook Usage')
plt.show()

## 3. VQ-VAE 编码器和解码器

In [ ]:
from fourm import VQVAEEncoder, VQVAEDecoder

encoder = VQVAEEncoder(in_channels=3, codebook_dim=128)
decoder = VQVAEDecoder(out_channels=3, codebook_dim=128)

x = torch.randn(2, 3, 256, 256)
z = encoder(x)
recon = decoder(z)

print(f'输入: {x.shape}')
print(f'编码: {z.shape}')
print(f'重建: {recon.shape}')

## 4. 多模态 Tokenizer

In [ ]:
from fourm import ModalityTokenizer

config = FourMConfig(image_size=64, modalities=['rgb', 'depth'])
tokenizer = ModalityTokenizer(config)

rgb = torch.randn(2, 3, 64, 64)
depth = torch.randn(2, 1, 64, 64)

tokens_rgb, _ = tokenizer.tokenize(rgb, 'rgb')
tokens_depth, _ = tokenizer.tokenize(depth, 'depth')

print(f'RGB tokens: {tokens_rgb.shape}')
print(f'Depth tokens: {tokens_depth.shape}')

## 5. Transformer 编码器

In [ ]:
from fourm import FourMEncoder

enc_config = FourMConfig(codebook_size=512, d_model=256, n_heads=4, n_encoder_layers=4, modalities=['rgb', 'depth'])
encoder = FourMEncoder(enc_config)

tokens = {'rgb': torch.randint(0, 512, (2, 16)), 'depth': torch.randint(0, 512, (2, 16))}
output = encoder(tokens)
print(f'编码器输出: {output.shape}')

In [ ]:
# 带 mask 的编码
mask_dict = {'rgb': torch.rand(2, 16) > 0.5, 'depth': torch.rand(2, 16) > 0.5}
output_masked = encoder(tokens, mask_dict)
print(f'Masked 编码器输出: {output_masked.shape}')

## 6. Transformer 解码器

In [ ]:
from fourm import FourMDecoder

decoder = FourMDecoder(enc_config)
encoder_output = torch.randn(2, 32, 256)
target_tokens = torch.randint(0, 512, (2, 16))

logits = decoder(encoder_output, target_tokens, target_modality_idx=0)
print(f'解码器 logits: {logits.shape}')

In [ ]:
# 自回归生成
generated = decoder.generate(encoder_output, target_modality_idx=0, max_len=16)
print(f'生成的 tokens: {generated.shape}')

## 7. 完整 4M 模型

In [ ]:
model = create_fourm_model('tiny')
n_params = sum(p.numel() for p in model.parameters())
print(f'Tiny 模型参数量: {n_params/1e6:.2f}M')

In [ ]:
# 前向传播
inputs = {'rgb': torch.randn(2, 3, 64, 64), 'depth': torch.randn(2, 1, 64, 64)}
loss, loss_dict = model(inputs, target_modality='rgb')
print(f'总损失: {loss.item():.4f}')
print(f'重建损失: {loss_dict["recon"].item():.4f}')
print(f'VQ 损失: {loss_dict["vq"].item():.4f}')

## 8. 模型参数统计

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('模型组件参数:')
print(f'  Tokenizer: {count_params(model.tokenizer)/1e6:.2f}M')
print(f'  Encoder: {count_params(model.encoder)/1e6:.2f}M')
print(f'  Decoder: {count_params(model.decoder)/1e6:.2f}M')

## 9. 总结

4M 的核心优势:
- 统一的多模态表示
- 任意模态到任意模态生成
- 可扩展到更多模态